In [43]:
import numpy as np
from scipy.optimize import linprog
import pandas as pd # Import pandas

# --- Data Setup ---
# Energy sources: [coal, oil, natural_gas, hydro, nuclear, solar, wind, biomass, storage]
costs = np.array([4.0, 8.5, 6.0, 4.5, 6.5, 2.5, 3.0, 4.5, 6.0])  # INR/kWh
emissions = np.array([0.82, 0.78, 0.45, 0.0, 0.0, 0.0, 0.0, 0.05, 0.0])  # kg CO2/kWh
sources = ['Coal', 'Oil', 'Natural Gas', 'Hydro', 'Nuclear', 'Solar', 'Wind', 'Biomass', 'Storage']

# Share bounds (as fractions of D)
min_shares = np.array([0.10, 0.01, 0.01, 0.07, 0.08, 0.25, 0.15, 0.02, 0.05])
max_shares = np.array([0.29, 0.05, 0.05, 0.10, 0.12, 0.40, 0.25, 0.05, 0.10])

# Total Demand (kWh)
D = 12778.67267* 10e9

# Weighting factor for the objective function (lambda=0 means pure emissions minimization)
lambda_ = 0

# --- Linear Programming Setup ---

# Objective: weighted sum of cost and emissions
# Since lambda_ = 0, the objective is to minimize total emissions: c = emissions
c = lambda_ * costs + (1 - lambda_) * emissions

# Equality constraint: total supply = D (Sum(S_j) = D)
A_eq = [np.ones(9)]
b_eq = [D]

# Inequality constraints: min and max shares for each source (S_j <= max_j*D and S_j >= min_j*D)
A_ub = []
b_ub = []
for i in range(9):
    # S_j <= max_share_j * D
    row = np.zeros(9)
    row[i] = 1
    A_ub.append(row)
    b_ub.append(max_shares[i] * D)
    
    # -S_j <= -min_share_j * D  --> S_j >= min_share_j * D
    row = np.zeros(9)
    row[i] = -1
    A_ub.append(row)
    b_ub.append(-min_shares[i] * D)

# Bounds for each variable (all >= 0)
bounds = [(0, None)] * 9

# --- Solve ---
# The default method 'highs' is generally efficient.
res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

# --- Results Presentation ---
if res.success:
    # Calculate shares and total metrics
    optimal_supply = res.x
    optimal_shares = optimal_supply / D
    total_cost = np.dot(costs, optimal_supply)
    total_emissions = np.dot(emissions, optimal_supply)
    
    # Create the DataFrame
    results_df = pd.DataFrame({
        'Source': sources,
        'Supply (kWh)': optimal_supply,
        'Optimal Share': optimal_shares,
        'Min Share': min_shares,
        'Max Share': max_shares,
        'Cost (INR/kWh)': costs,
        'Emissions (kg CO2/kWh)': emissions
    })
    
    # Format for better readability
    pd.options.display.float_format = '{:,.2f}'.format
    
    """print("## Energy Mix Optimization Results (Emissions Minimization, $\\lambda = 0$)\n")
    print(results_df)

    print("\n" + "-"*50)
    print(f"Total Demand (D): {D:,.2f} kWh")
    print(f"Total Cost: {total_cost:,.2f} INR")
    print(f"Total Emissions: {total_emissions:,.2f} kg CO2")
    print("-"*50)
"""
else:
    print("Optimization failed:", res.message)
    
    
results_df['Supply (GWh)']=results_df['Supply (kWh)']/1e6
results_df

,Source,Supply (kWh),Optimal Share,Min Share,Max Share,Cost (INR/kWh),Emissions (kg CO2/kWh),Supply (GWh)
0,Coal,"12,778,672,670,000.00",0.10,0.10,0.29,4.00,0.82,"12,778,672.67"
1,Oil,"1,277,867,267,000.00",0.01,0.01,0.05,8.50,0.78,"1,277,867.27"
2,Natural Gas,"1,277,867,267,000.00",0.01,0.01,0.05,6.00,0.45,"1,277,867.27"
3,Hydro,"8,945,070,869,000.00",0.07,0.07,0.10,4.50,0.00,"8,945,070.87"
4,Nuclear,"10,222,938,136,000.00",0.08,0.08,0.12,6.50,0.00,"10,222,938.14"
5,Solar,"46,003,221,612,000.00",0.36,0.25,0.40,2.50,0.00,"46,003,221.61"
6,Wind,"31,946,681,675,000.00",0.25,0.15,0.25,3.00,0.00,"31,946,681.68"
7,Biomass,"2,555,734,534,000.00",0.02,0.02,0.05,4.50,0.05,"2,555,734.53"
8,Storage,"12,778,672,670,000.00",0.10,0.05,0.10,6.00,0.00,"12,778,672.67"


In [53]:
import numpy as np
from scipy.optimize import linprog
import pandas as pd # Import pandas

# --- Data Setup ---
# Energy sources: [coal, oil, natural_gas, hydro, nuclear, solar, wind, biomass, storage]
costs = np.array([4.0, 8.5, 6.0, 4.5, 6.5, 2.5, 3.0, 4.5, 6.0])  # INR/kWh
emissions = np.array([0.82, 0.78, 0.45, 0.0, 0.0, 0.0, 0.0, 0.05, 0.0])  # kg CO2/kWh
sources = ['Coal', 'Oil', 'Natural Gas', 'Hydro', 'Nuclear', 'Solar', 'Wind', 'Biomass', 'Storage']

# Share bounds (as fractions of D)
min_shares = np.array([0.10, 0.01, 0.01, 0.07, 0.08, 0.25, 0.15, 0.02, 0.05])
max_shares = np.array([0.29, 0.05, 0.05, 0.10, 0.12, 0.40, 0.25, 0.05, 0.10])

# Total Demand (kWh)
D = 12778.67267* 10e9

# Weighting factor for the objective function (lambda=0 means pure emissions minimization)
lambda_ = 1

# --- Linear Programming Setup ---

# Objective: weighted sum of cost and emissions
# Since lambda_ = 0, the objective is to minimize total emissions: c = emissions
c = lambda_ * costs + (1 - lambda_) * emissions

# Equality constraint: total supply = D (Sum(S_j) = D)
A_eq = [np.ones(9)]
b_eq = [D]

# Inequality constraints: min and max shares for each source (S_j <= max_j*D and S_j >= min_j*D)
A_ub = []
b_ub = []
for i in range(9):
    # S_j <= max_share_j * D
    row = np.zeros(9)
    row[i] = 1
    A_ub.append(row)
    b_ub.append(max_shares[i] * D)
    
    # -S_j <= -min_share_j * D  --> S_j >= min_share_j * D
    row = np.zeros(9)
    row[i] = -1
    A_ub.append(row)
    b_ub.append(-min_shares[i] * D)

# Bounds for each variable (all >= 0)
bounds = [(0, None)] * 9

# --- Solve ---
# The default method 'highs' is generally efficient.
res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

# --- Results Presentation ---
if res.success:
    # Calculate shares and total metrics
    optimal_supply = res.x
    optimal_shares = optimal_supply / D
    total_cost = np.dot(costs, optimal_supply)
    total_emissions = np.dot(emissions, optimal_supply)
    
    # Create the DataFrame
    results_df = pd.DataFrame({
        'Source': sources,
        'Supply (kWh)': optimal_supply,
        'Optimal Share': optimal_shares,
        'Min Share': min_shares,
        'Max Share': max_shares,
        'Cost (INR/kWh)': costs,
        'Emissions (kg CO2/kWh)': emissions
    })
    
    # Format for better readability
    pd.options.display.float_format = '{:,.2f}'.format
    
    """print("## Energy Mix Optimization Results (Emissions Minimization, $\\lambda = 0$)\n")
    print(results_df)

    print("\n" + "-"*50)
    print(f"Total Demand (D): {D:,.2f} kWh")
    print(f"Total Cost: {total_cost:,.2f} INR")
    print(f"Total Emissions: {total_emissions:,.2f} kg CO2")
    print("-"*50)
"""
else:
    print("Optimization failed:", res.message)
    
    
results_df['Supply (GWh)']=results_df['Supply (kWh)']/1e6
results_df

,Source,Supply (kWh),Optimal Share,Min Share,Max Share,Cost (INR/kWh),Emissions (kg CO2/kWh),Supply (GWh)
0,Coal,"14,056,539,937,000.00",0.11,0.10,0.29,4.00,0.82,"14,056,539.94"
1,Oil,"1,277,867,267,000.00",0.01,0.01,0.05,8.50,0.78,"1,277,867.27"
2,Natural Gas,"1,277,867,267,000.00",0.01,0.01,0.05,6.00,0.45,"1,277,867.27"
3,Hydro,"8,945,070,869,000.00",0.07,0.07,0.10,4.50,0.00,"8,945,070.87"
4,Nuclear,"10,222,938,136,000.00",0.08,0.08,0.12,6.50,0.00,"10,222,938.14"
5,Solar,"51,114,690,680,000.00",0.40,0.25,0.40,2.50,0.00,"51,114,690.68"
6,Wind,"31,946,681,675,000.00",0.25,0.15,0.25,3.00,0.00,"31,946,681.68"
7,Biomass,"2,555,734,534,000.00",0.02,0.02,0.05,4.50,0.05,"2,555,734.53"
8,Storage,"6,389,336,335,000.00",0.05,0.05,0.10,6.00,0.00,"6,389,336.33"


In [54]:
import numpy as np
from scipy.optimize import linprog
import pandas as pd # Import pandas

# --- Data Setup ---
# Energy sources: [coal, oil, natural_gas, hydro, nuclear, solar, wind, biomass, storage]
costs = np.array([4.0, 8.5, 6.0, 4.5, 6.5, 2.5, 3.0, 4.5, 6.0])  # INR/kWh
emissions = np.array([0.82, 0.78, 0.45, 0.0, 0.0, 0.0, 0.0, 0.05, 0.0])  # kg CO2/kWh
sources = ['Coal', 'Oil', 'Natural Gas', 'Hydro', 'Nuclear', 'Solar', 'Wind', 'Biomass', 'Storage']

# Share bounds (as fractions of D)
min_shares = np.array([0.10, 0.01, 0.01, 0.07, 0.08, 0.25, 0.15, 0.02, 0.05])
max_shares = np.array([0.29, 0.05, 0.05, 0.10, 0.12, 0.40, 0.25, 0.05, 0.10])

# Total Demand (kWh)
D = 12778.67267* 10e9

# Weighting factor for the objective function (lambda=0 means pure emissions minimization)
lambda_ = 0.5

# --- Linear Programming Setup ---

# Objective: weighted sum of cost and emissions
# Since lambda_ = 0, the objective is to minimize total emissions: c = emissions
c = lambda_ * costs + (1 - lambda_) * emissions

# Equality constraint: total supply = D (Sum(S_j) = D)
A_eq = [np.ones(9)]
b_eq = [D]

# Inequality constraints: min and max shares for each source (S_j <= max_j*D and S_j >= min_j*D)
A_ub = []
b_ub = []
for i in range(9):
    # S_j <= max_share_j * D
    row = np.zeros(9)
    row[i] = 1
    A_ub.append(row)
    b_ub.append(max_shares[i] * D)
    
    # -S_j <= -min_share_j * D  --> S_j >= min_share_j * D
    row = np.zeros(9)
    row[i] = -1
    A_ub.append(row)
    b_ub.append(-min_shares[i] * D)

# Bounds for each variable (all >= 0)
bounds = [(0, None)] * 9

# --- Solve ---
# The default method 'highs' is generally efficient.
res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

# --- Results Presentation ---
if res.success:
    # Calculate shares and total metrics
    optimal_supply = res.x
    optimal_shares = optimal_supply / D
    total_cost = np.dot(costs, optimal_supply)
    total_emissions = np.dot(emissions, optimal_supply)
    
    # Create the DataFrame
    results_df = pd.DataFrame({
        'Source': sources,
        'Supply (kWh)': optimal_supply,
        'Optimal Share': optimal_shares,
        'Min Share': min_shares,
        'Max Share': max_shares,
        'Cost (INR/kWh)': costs,
        'Emissions (kg CO2/kWh)': emissions
    })
    
    # Format for better readability
    pd.options.display.float_format = '{:,.2f}'.format
    
    """print("## Energy Mix Optimization Results (Emissions Minimization, $\\lambda = 0$)\n")
    print(results_df)

    print("\n" + "-"*50)
    print(f"Total Demand (D): {D:,.2f} kWh")
    print(f"Total Cost: {total_cost:,.2f} INR")
    print(f"Total Emissions: {total_emissions:,.2f} kg CO2")
    print("-"*50)
"""
else:
    print("Optimization failed:", res.message)
    
    
results_df['Supply (GWh)']=results_df['Supply (kWh)']/1e6
results_df

,Source,Supply (kWh),Optimal Share,Min Share,Max Share,Cost (INR/kWh),Emissions (kg CO2/kWh),Supply (GWh)
0,Coal,"12,778,672,670,000.00",0.10,0.10,0.29,4.00,0.82,"12,778,672.67"
1,Oil,"1,277,867,267,000.00",0.01,0.01,0.05,8.50,0.78,"1,277,867.27"
2,Natural Gas,"1,277,867,267,000.00",0.01,0.01,0.05,6.00,0.45,"1,277,867.27"
3,Hydro,"10,222,938,136,000.00",0.08,0.07,0.10,4.50,0.00,"10,222,938.14"
4,Nuclear,"10,222,938,136,000.00",0.08,0.08,0.12,6.50,0.00,"10,222,938.14"
5,Solar,"51,114,690,680,000.00",0.40,0.25,0.40,2.50,0.00,"51,114,690.68"
6,Wind,"31,946,681,675,000.00",0.25,0.15,0.25,3.00,0.00,"31,946,681.68"
7,Biomass,"2,555,734,534,000.00",0.02,0.02,0.05,4.50,0.05,"2,555,734.53"
8,Storage,"6,389,336,335,000.00",0.05,0.05,0.10,6.00,0.00,"6,389,336.33"


In [55]:
import numpy as np
from scipy.optimize import linprog
import pandas as pd # Import pandas

# --- Data Setup ---
# Energy sources: [coal, oil, natural_gas, hydro, nuclear, solar, wind, biomass, storage]
costs = np.array([4.0, 8.5, 6.0, 4.5, 6.5, 2.5, 3.0, 4.5, 6.0])  # INR/kWh
emissions = np.array([0.82, 0.78, 0.45, 0.0, 0.0, 0.0, 0.0, 0.05, 0.0])  # kg CO2/kWh
sources = ['Coal', 'Oil', 'Natural Gas', 'Hydro', 'Nuclear', 'Solar', 'Wind', 'Biomass', 'Storage']

# Share bounds (as fractions of D)
min_shares = np.array([0.10, 0.01, 0.01, 0.07, 0.08, 0.25, 0.15, 0.02, 0.05])
max_shares = np.array([0.29, 0.05, 0.05, 0.10, 0.12, 0.40, 0.25, 0.05, 0.10])

# Total Demand (kWh)
D = 12778.67267* 10e9

# Weighting factor for the objective function (lambda=0 means pure emissions minimization)
lambda_ = 0.25

# --- Linear Programming Setup ---

# Objective: weighted sum of cost and emissions
# Since lambda_ = 0, the objective is to minimize total emissions: c = emissions
c = lambda_ * costs + (1 - lambda_) * emissions

# Equality constraint: total supply = D (Sum(S_j) = D)
A_eq = [np.ones(9)]
b_eq = [D]

# Inequality constraints: min and max shares for each source (S_j <= max_j*D and S_j >= min_j*D)
A_ub = []
b_ub = []
for i in range(9):
    # S_j <= max_share_j * D
    row = np.zeros(9)
    row[i] = 1
    A_ub.append(row)
    b_ub.append(max_shares[i] * D)
    
    # -S_j <= -min_share_j * D  --> S_j >= min_share_j * D
    row = np.zeros(9)
    row[i] = -1
    A_ub.append(row)
    b_ub.append(-min_shares[i] * D)

# Bounds for each variable (all >= 0)
bounds = [(0, None)] * 9

# --- Solve ---
# The default method 'highs' is generally efficient.
res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

# --- Results Presentation ---
if res.success:
    # Calculate shares and total metrics
    optimal_supply = res.x
    optimal_shares = optimal_supply / D
    total_cost = np.dot(costs, optimal_supply)
    total_emissions = np.dot(emissions, optimal_supply)
    
    # Create the DataFrame
    results_df = pd.DataFrame({
        'Source': sources,
        'Supply (kWh)': optimal_supply,
        'Optimal Share': optimal_shares,
        'Min Share': min_shares,
        'Max Share': max_shares,
        'Cost (INR/kWh)': costs,
        'Emissions (kg CO2/kWh)': emissions
    })
    
    # Format for better readability
    pd.options.display.float_format = '{:,.2f}'.format
    
    """print("## Energy Mix Optimization Results (Emissions Minimization, $\\lambda = 0$)\n")
    print(results_df)

    print("\n" + "-"*50)
    print(f"Total Demand (D): {D:,.2f} kWh")
    print(f"Total Cost: {total_cost:,.2f} INR")
    print(f"Total Emissions: {total_emissions:,.2f} kg CO2")
    print("-"*50)
"""
else:
    print("Optimization failed:", res.message)
    
    
results_df['Supply (GWh)']=results_df['Supply (kWh)']/1e6
results_df

,Source,Supply (kWh),Optimal Share,Min Share,Max Share,Cost (INR/kWh),Emissions (kg CO2/kWh),Supply (GWh)
0,Coal,"12,778,672,670,000.00",0.10,0.10,0.29,4.00,0.82,"12,778,672.67"
1,Oil,"1,277,867,267,000.00",0.01,0.01,0.05,8.50,0.78,"1,277,867.27"
2,Natural Gas,"1,277,867,267,000.00",0.01,0.01,0.05,6.00,0.45,"1,277,867.27"
3,Hydro,"10,222,938,136,000.00",0.08,0.07,0.10,4.50,0.00,"10,222,938.14"
4,Nuclear,"10,222,938,136,000.00",0.08,0.08,0.12,6.50,0.00,"10,222,938.14"
5,Solar,"51,114,690,680,000.00",0.40,0.25,0.40,2.50,0.00,"51,114,690.68"
6,Wind,"31,946,681,675,000.00",0.25,0.15,0.25,3.00,0.00,"31,946,681.68"
7,Biomass,"2,555,734,534,000.00",0.02,0.02,0.05,4.50,0.05,"2,555,734.53"
8,Storage,"6,389,336,335,000.00",0.05,0.05,0.10,6.00,0.00,"6,389,336.33"


In [56]:
import numpy as np
from scipy.optimize import linprog
import pandas as pd # Import pandas

# --- Data Setup ---
# Energy sources: [coal, oil, natural_gas, hydro, nuclear, solar, wind, biomass, storage]
costs = np.array([4.0, 8.5, 6.0, 4.5, 6.5, 2.5, 3.0, 4.5, 6.0])  # INR/kWh
emissions = np.array([0.82, 0.78, 0.45, 0.0, 0.0, 0.0, 0.0, 0.05, 0.0])  # kg CO2/kWh
sources = ['Coal', 'Oil', 'Natural Gas', 'Hydro', 'Nuclear', 'Solar', 'Wind', 'Biomass', 'Storage']

# Share bounds (as fractions of D)
min_shares = np.array([0.10, 0.01, 0.01, 0.07, 0.08, 0.25, 0.15, 0.02, 0.05])
max_shares = np.array([0.29, 0.05, 0.05, 0.10, 0.12, 0.40, 0.25, 0.05, 0.10])

# Total Demand (kWh)
D = 12778.67267* 10e9

# Weighting factor for the objective function (lambda=0 means pure emissions minimization)
lambda_ = 0.75


# --- Linear Programming Setup ---

# Objective: weighted sum of cost and emissions
# Since lambda_ = 0, the objective is to minimize total emissions: c = emissions
c = lambda_ * costs + (1 - lambda_) * emissions

# Equality constraint: total supply = D (Sum(S_j) = D)
A_eq = [np.ones(9)]
b_eq = [D]

# Inequality constraints: min and max shares for each source (S_j <= max_j*D and S_j >= min_j*D)
A_ub = []
b_ub = []
for i in range(9):
    # S_j <= max_share_j * D
    row = np.zeros(9)
    row[i] = 1
    A_ub.append(row)
    b_ub.append(max_shares[i] * D)
    
    # -S_j <= -min_share_j * D  --> S_j >= min_share_j * D
    row = np.zeros(9)
    row[i] = -1
    A_ub.append(row)
    b_ub.append(-min_shares[i] * D)

# Bounds for each variable (all >= 0)
bounds = [(0, None)] * 9

# --- Solve ---
# The default method 'highs' is generally efficient.
res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

# --- Results Presentation ---
if res.success:
    # Calculate shares and total metrics
    optimal_supply = res.x
    optimal_shares = optimal_supply / D
    total_cost = np.dot(costs, optimal_supply)
    total_emissions = np.dot(emissions, optimal_supply)
    
    # Create the DataFrame
    results_df = pd.DataFrame({
        'Source': sources,
        'Supply (kWh)': optimal_supply,
        'Optimal Share': optimal_shares,
        'Min Share': min_shares,
        'Max Share': max_shares,
        'Cost (INR/kWh)': costs,
        'Emissions (kg CO2/kWh)': emissions
    })
    
    # Format for better readability
    pd.options.display.float_format = '{:,.2f}'.format
    
    """print("## Energy Mix Optimization Results (Emissions Minimization, $\\lambda = 0$)\n")
    print(results_df)

    print("\n" + "-"*50)
    print(f"Total Demand (D): {D:,.2f} kWh")
    print(f"Total Cost: {total_cost:,.2f} INR")
    print(f"Total Emissions: {total_emissions:,.2f} kg CO2")
    print("-"*50)
"""
else:
    print("Optimization failed:", res.message)
    
    
results_df['Supply (GWh)']=results_df['Supply (kWh)']/1e6
results_df

,Source,Supply (kWh),Optimal Share,Min Share,Max Share,Cost (INR/kWh),Emissions (kg CO2/kWh),Supply (GWh)
0,Coal,"14,056,539,937,000.00",0.11,0.10,0.29,4.00,0.82,"14,056,539.94"
1,Oil,"1,277,867,267,000.00",0.01,0.01,0.05,8.50,0.78,"1,277,867.27"
2,Natural Gas,"1,277,867,267,000.00",0.01,0.01,0.05,6.00,0.45,"1,277,867.27"
3,Hydro,"8,945,070,869,000.00",0.07,0.07,0.10,4.50,0.00,"8,945,070.87"
4,Nuclear,"10,222,938,136,000.00",0.08,0.08,0.12,6.50,0.00,"10,222,938.14"
5,Solar,"51,114,690,680,000.00",0.40,0.25,0.40,2.50,0.00,"51,114,690.68"
6,Wind,"31,946,681,675,000.00",0.25,0.15,0.25,3.00,0.00,"31,946,681.68"
7,Biomass,"2,555,734,534,000.00",0.02,0.02,0.05,4.50,0.05,"2,555,734.53"
8,Storage,"6,389,336,335,000.00",0.05,0.05,0.10,6.00,0.00,"6,389,336.33"
